# Day 5 Lab: Delta Live Tables (DLT) - Medallion Pipeline

## Objectives
- Build Bronze, Silver, Gold layers using DLT
- Apply data quality checks
- Understand declarative pipelines


## Step 1: Setup
Upload orders.csv to DBFS: dbfs:/FileStore/labs/orders.csv

In [ ]:
import dlt
from pyspark.sql.functions import *

In [ ]:
@dlt.table(name='bronze_orders')
def bronze_orders():
    return spark.read.format('csv').option('header', True).load('dbfs:/FileStore/labs/orders.csv')

In [ ]:
@dlt.table(name='silver_orders')
@dlt.expect('valid_amount', 'amount IS NOT NULL AND amount > 0')
def silver_orders():
    df = dlt.read('bronze_orders')
    return df.filter(col('status') == 'completed').withColumn('amount', col('amount').cast('int'))

In [ ]:
@dlt.table(name='gold_customer_spend')
def gold_customer_spend():
    df = dlt.read('silver_orders')
    return df.groupBy('customer_id').agg(sum('amount').alias('total_spent'))

In [ ]:
%sql
SELECT * FROM gold_customer_spend ORDER BY total_spent DESC